In [1]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version

model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
               total        used        free      shared  buff/cache   available
Mem:            12Gi       632Mi       9.1Gi       1.0Mi       2.9Gi        11Gi
Swap:             0B          0B          0B
Python 3.12.13


In [2]:
%cd /content
!rm -rf networkx
!git clone https://github.com/networkx/networkx.git
%cd /content/networkx
!git checkout 9210d9c5bb9875caae0c7be2214abebfdd9255d2
!pip install -e ".[default]" -q
!git rev-parse HEAD

/content
Cloning into 'networkx'...
remote: Enumerating objects: 74244, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 74244 (delta 124), reused 40 (delta 37), pack-reused 74041 (from 4)
Receiving objects: 100% (74244/74244), 94.09 MiB | 18.10 MiB/s, done.
Resolving deltas: 100% (51588/51588), done.
/content/networkx
Note: switching to '9210d9c5bb9875caae0c7be2214abebfdd9255d2'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 9210d9c5b Desi

## Repo & Workload Description

**Repo:** networkx/networkx  
**Baseline commit:** 9210d9c5bb9875caae0c7be2214abebfdd9255d2 (tag: networkx-3.1)  
**License:** BSD-3-Clause

**Slow workload:** `networkx.betweenness_centrality()` on a dense random graph (1500 nodes, 50000 edges)

**Why this workload is meaningful:**  
Betweenness centrality is one of the most widely used graph analysis algorithms. It measures how often a node appears on shortest paths between other nodes. It is used in social network analysis, transportation routing, and biological network research. NetworkX's implementation uses Brandes' algorithm, which is O(VE) on a graph with 1500 nodes and 50000 edges, this requires millions of BFS traversals entirely in pure Python, making it a real and significant bottleneck.

**How I found it:**  
The pure-Python implementation of betweenness centrality in NetworkX is a well-documented bottleneck in the community (GitHub issues, Stack Overflow, academic benchmarks).

In [3]:
import sys
sys.path.insert(0, '/content/networkx')
import networkx as nx
import json

print(f"NetworkX version: {nx.__version__}")

# Build the benchmark graph
G = nx.gnm_random_graph(1500, 50000, seed=42)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Run workload once and save golden output
result = nx.betweenness_centrality(G, normalized=True)

# Save golden output (keys are ints, convert to strings for JSON)
golden = {str(k): v for k, v in result.items()}
with open("/content/golden_output.json", "w") as f:
    json.dump(golden, f)

print(f"Golden output saved. Sample: node 0 = {result[0]:.8f}")

NetworkX version: 3.1
Graph: 1500 nodes, 50000 edges
Golden output saved. Sample: node 0 = 0.00059186


In [4]:
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest",
     "networkx/algorithms/centrality/tests/test_betweenness_centrality.py",
     "-v", "--tb=short", "-q"],
    capture_output=True, text=True,
    cwd="/content/networkx"
)

print(result.stdout[-4000:])
print(result.stderr[-1000:])

# Save pass-set
with open("/content/baseline_passset.txt", "w") as f:
    f.write(result.stdout)

print("\nPass-set saved to /content/baseline_passset.txt")

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/networkx
plugins: typeguard-4.5.1, anyio-4.13.0, langsmith-0.7.34
collected 41 items

networkx/algorithms/centrality/tests/test_betweenness_centrality.py .... [  9%]
.....................................                                    [100%]

============================== 41 passed in 0.22s ==============================



Pass-set saved to /content/baseline_passset.txt


In [5]:
import time
import numpy as np
import networkx as nx

# Rebuild graph with same seed (reproducible)
G = nx.gnm_random_graph(1500, 50000, seed=42)

print("Running 2 warmup runs...")
for i in range(2):
    _ = nx.betweenness_centrality(G, normalized=True)
    print(f"  Warmup {i+1} done")

print("\nRunning 7 measured runs...")
times = []
for i in range(7):
    start = time.perf_counter()
    _ = nx.betweenness_centrality(G, normalized=True)
    elapsed = time.perf_counter() - start
    times.append(elapsed)
    print(f"  Run {i+1}: {elapsed:.3f}s")

median_time = float(np.median(times))
iqr = float(np.percentile(times, 75) - np.percentile(times, 25))

print(f"\n--- Baseline Timing Results ---")
print(f"All runs (s): {[round(t, 3) for t in times]}")
print(f"Median: {median_time:.3f}s")
print(f"IQR:    {iqr:.3f}s")

# Save timing results for later use in tests.ipynb
import json
timing = {"median": median_time, "iqr": iqr, "n_warmup": 2, "n_measured": 7}
with open("/content/baseline_timing.json", "w") as f:
    json.dump(timing, f)
print("\nTiming saved to /content/baseline_timing.json")

Running 2 warmup runs...
  Warmup 1 done
  Warmup 2 done

Running 7 measured runs...
  Run 1: 26.017s
  Run 2: 26.341s
  Run 3: 26.559s
  Run 4: 25.801s
  Run 5: 25.738s
  Run 6: 25.563s
  Run 7: 25.643s

--- Baseline Timing Results ---
All runs (s): [26.017, 26.341, 26.559, 25.801, 25.738, 25.563, 25.643]
Median: 25.801s
IQR:    0.488s

Timing saved to /content/baseline_timing.json
